In [1]:
"""
Prediction Script - matches train_model.py preprocessing exactly.
Reuses the saved encoders, scaler, and locked column order so nothing
gets silently misaligned before it hits the model.
"""

import pickle
import numpy as np
import pandas as pd
import tensorflow as tf


# ---------------------------------------------------------------------------
# 1. LOAD MODEL + SAVED PREPROCESSING ARTIFACTS
# ---------------------------------------------------------------------------
model = tf.keras.models.load_model('model.keras')

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

# ---------------------------------------------------------------------------
# 2. SAMPLE INPUT
# ---------------------------------------------------------------------------
input_data = {
    'CreditScore': 400,
    'Geography': 'Germany',
    'Gender': 'Female',
    'Age': 35,
    'Tenure': 3,
    'Balance': 100000,
    'NumOfProducts': 3,
    'HasCrCard': 1,
    'IsActiveMember': 0,
    'EstimatedSalary': 75000
}

input_df = pd.DataFrame([input_data])

# ---------------------------------------------------------------------------
# 3. ENCODE  (identical steps and identical fitted encoders as training)
# ---------------------------------------------------------------------------
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)

# ---------------------------------------------------------------------------
# 4. REORDER COLUMNS TO MATCH TRAINING EXACTLY
#    (this is the step the original code skipped -- without it, a scaler
#    fit on one column order silently scales the wrong values)
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# 5. SCALE + PREDICT
# ---------------------------------------------------------------------------
input_scaled = scaler.transform(input_df)
prediction = model.predict(input_scaled)
prediction_proba = prediction[0][0]

print(f"Churn probability: {prediction_proba:.4f}")
print("Prediction:", "Customer will churn" if prediction_proba > 0.5 else "Customer will stay")


2026-09-18 12:56:53.290543: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-09-18 12:56:53.290584: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-18 12:56:53.290596: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-18 12:56:53.290852: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-18 12:56:53.291190: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


1/1 [==============================] - 0s 138ms/step
Churn probability: 0.6386
Prediction: Customer will churn


2026-09-18 12:56:55.027513: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
